# `agent/planner.py` — Validation Walkthrough

Exercises the full retrieve → plan pipeline against the indexed FE 501s manual.

The planner receives chunks already sorted in prerequisite-first order by the retrieval
layer, formats them into a structured prompt, and calls Claude to produce a numbered,
section-organised repair plan.  Torque specs and image paths come directly from chunk
metadata — no second Claude call is needed for those.

In [1]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

import anthropic
from openai import OpenAI
from agent.retrieval import retrieve, retrieve_from_section, open_collection
from agent.planner import plan, _build_context, _collect_torque_specs

MANUAL = "FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du"

col    = open_collection("../data/chroma")
openai = OpenAI()
claude = anthropic.Anthropic()

print(f"Collection: {col.count()} chunks indexed")

Collection: 600 chunks indexed


## 1. Prompt context inspection

Before calling Claude, inspect what `_build_context` produces.  This is the text
Claude receives as the manual sections block.

In [2]:
chunks = retrieve_from_section("7.2", col, manual=MANUAL)

print(f"{len(chunks)} chunks retrieved for section 7.2\n")
for c in chunks:
    print(f"  depth={c.depth}  {c.section}  [{c.phase or 'full'}]")

print("\n--- Context block (first 800 chars) ---")
print(_build_context(chunks)[:800])

1 chunks retrieved for section 7.2

  depth=0  7.2  [full]

--- Context block (first 800 chars) ---
=== Section 7.2: Adjusting the handlebar position (target) ===
7.2 Adjusting the handlebar position
WARNING
Danger of accidents A repaired handlebar poses a safety risk.
If the handlebar is bent or straightened, the material becomes fatigued. The handlebar may break as a
result.
‒ Change the handlebar if the handlebar is damaged or bent.
W00323-10
‒ Remove (1) screws. Take off the handlebar clamp. Remove
the handlebar and lay it to one side.
Protect the components against damage by covering
them.
Do not kink the cables or lines.
‒ Remove (2) screws. Take off the handlebar support.
‒ Place the handlebar mount in the required position. Mount
and tighten screws (2).
Screw, handlebar mount
M10 40 Nm
(29.5 ft⋅lbf)
Loctite® 243
Position the handlebar support so that it is even.
‒ Position the ha


## 2. Simple plan — handlebar adjustment

Section 7.2 has no prerequisite sections (it's a straightforward adjustment), so the
result is a single-section plan.  Good baseline to verify format and torque annotation.

In [3]:
result = plan("adjust the handlebar position", chunks, claude)

print(f"Sections used: {result.sections_used}")
print(f"Torque specs:  {len(result.torque_specs)}")
print(f"Image paths:   {len(result.image_paths)}")
print()
print(result.text)

Sections used: ['7.2']
Torque specs:  2
Image paths:   3

# Handlebar Position Adjustment Plan

---

## Section 7.2: Adjusting the Handlebar Position

> ⚠️ **WARNING:** Never bend or straighten a damaged handlebar — material fatigue may cause it to break. **Replace the handlebar if it is bent or damaged.**

---

**1. Remove the handlebar clamp.**
Remove screws **(1)** and take off the handlebar clamp.

**2. Remove the handlebar.**
Carefully remove the handlebar and lay it to one side.
- Cover nearby components to protect them from damage.
- Do **not** kink any cables or lines.

**3. Remove the handlebar support.**
Remove screws **(2)** and take off the handlebar support.

**4. Reposition the handlebar mount.**
Place the handlebar mount in the required position.

**5. Reinstall the handlebar support.**
Mount screws **(2)** and torque to **40 Nm (29.5 ft·lbf)** using **Loctite® 243**.
- Ensure the handlebar support is positioned evenly before final tightening.

**6. Reposition the handle

## 3. Multi-section plan — fork cartridge disassembly

Section 6.12 requires completing 6.10 and 6.11 first.  The retrieval layer surfaces
those as depth-1 prerequisites; the planner should sequence them correctly and produce
a multi-section plan.

In [4]:
chunks = retrieve_from_section("6.12", col, manual=MANUAL)

print(f"{len(chunks)} chunks retrieved\n")
for c in chunks:
    print(f"  depth={c.depth}  {c.section}  [{c.phase or 'full'}]  {c.section_title}")

result = plan("disassemble the fork cartridge", chunks, claude)

print(f"\nSections used: {result.sections_used}")
print(f"Torque specs:  {len(result.torque_specs)}")
print()
print(result.text)

7 chunks retrieved

  depth=1  6.10  [full]  Disassembling the fork legs
  depth=1  6.11  [full]  Disassembling the cartridge
  depth=1  6.11  [preparatory]  Disassembling the cartridge
  depth=1  6.11  [main]  Disassembling the cartridge
  depth=0  6.12  [full]  Disassembling the piston rod
  depth=0  6.12  [preparatory]  Disassembling the piston rod
  depth=0  6.12  [main]  Disassembling the piston rod

Sections used: ['6.10', '6.11', '6.12']
Torque specs:  0

# Fork Cartridge Disassembly Repair Plan

---

## Section 6.10: Disassembling the Fork Legs

> **Condition:** Fork legs have been removed. Operations are the same on both fork legs.

1. Make a note of the current positions of the rebound (1) and compression damping (2) adjusters.
2. Open the adjusters of both the rebound and compression damping completely.
3. Clamp the fork leg in the area of the lower triple clamp using the **Clamping Stand (T1403S)**.
4. Remove screw (3) and remove the compression damping adjuster.
5. Loosen 

## 4. Disambiguation flow — oil and filter change

The query matches both Ch.18 (engine-out assembly) and Ch.22 (routine maintenance).
After disambiguation the user picks Ch.22; `retrieve_from_section` scopes the plan
to the correct context.

In [5]:
from collections import defaultdict

# Step 1: retrieve candidates and detect multi-chapter result
candidates = retrieve("oil and filter change", col, openai, manual=MANUAL, n_results=5)
seeds = [c for c in candidates if c.depth == 0]

by_chapter: dict = defaultdict(list)
seen: set = set()
for c in seeds:
    if c.section not in seen:
        seen.add(c.section)
        by_chapter[(c.chapter_num, c.chapter_title)].append(c.section)

print("Candidates grouped by chapter:")
for (ch_num, ch_title), sections in sorted(by_chapter.items()):
    print(f"  Ch. {ch_num} — {ch_title}: {sections}")

print("\n→ User selects Ch. 22 (engine in frame)")

Candidates grouped by chapter:
  Ch. 18 — engine: ['18.3.3', '18.5.9']
  Ch. 22 — Lubrication system: ['22.3', '22.5']

→ User selects Ch. 22 (engine in frame)


In [6]:
# Step 2: scope to the chosen section and generate plan
scoped = retrieve_from_section("22.3", col, manual=MANUAL)
result = plan("change the engine oil and oil filter", scoped, claude)

print(f"Sections used: {result.sections_used}")
chapters_in_plan = {int(s.split('.')[0]) for s in result.sections_used}
print(f"Chapters in plan: {sorted(chapters_in_plan)}  (Ch.18 excluded)")
print()
print(result.text)

Sections used: ['8.4', '22.3']
Chapters in plan: [8, 22]  (Ch.18 excluded)

# Repair Plan: Engine Oil and Oil Filter Change

---

## Safety Precautions

1. Ensure the engine is at **operating temperature** before beginning — this allows oil to drain completely.
2. Wear suitable **protective clothing and safety gloves** to avoid scalding from hot oil.
3. In the event of scalding, rinse the affected area immediately with lukewarm water.
4. Prepare appropriate containers for waste oil and dispose of all oil, filters, and sealing parts in accordance with applicable environmental regulations.

---

## Preparatory Work

1. Remove the **skid plate** (refer to p. 57 of the manual).
2. Park the motorcycle on a **level surface**.

---

## Main Work

### Drain the Engine Oil

1. Position an appropriate **drain container** under the engine.
2. Remove **oil drain plug (1)** along with its magnet and seal ring; set aside the magnet.
3. Remove **screw plug (2)** with its O-ring.
   > ⚠️ Do **not** re

## 5. Torque spec and image path passthrough

Structured metadata comes from chunk fields, not parsed from Claude's text.  Verify
that specs and image paths round-trip correctly through the planner.

In [7]:
chunks = retrieve_from_section("7.2", col, manual=MANUAL)
result = plan("adjust the handlebar", chunks, claude)

if result.torque_specs:
    print(f"{'Bolt':<12} {'Nm':>6}  {'ft·lbf':>7}  Note")
    print("-" * 50)
    for s in result.torque_specs:
        print(f"{s['bolt']:<12} {s['nm']:>6}  {s['ftlbf']:>7}  {s['note']}")
else:
    print("No torque specs.")

print(f"\nImage paths ({len(result.image_paths)}):")
for p in result.image_paths:
    print(f"  {p}")

Bolt             Nm   ft·lbf  Note
--------------------------------------------------
M10            40.0     29.5  Loctite® 243
M8             20.0     14.8  Make sure the installed gap widths are even.

Image paths (3):
  data/images/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du/p0050_00.jpg
  data/images/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du/p0050_01.jpg
  data/images/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du/p0051_00.jpg
